# Convert merged model to MLC for WebLLM

Standalone notebook: pulls the merged HF weights produced by `train.ipynb`, runs `mlc_llm convert_weight` + `mlc_llm gen_config` with `q4f16_1` quantization, and pushes to a new HF repo. Output is consumable by `@mlc-ai/web-llm` (browser inference, WebGPU).

Runs on Kaggle T4 (cu122 wheels) but the conversion itself is mostly CPU-bound. Wall time is dominated by the upload at the end.


In [ ]:
# Install MLC nightly. Package names are CUDA-suffixed; cu122 matches Kaggle T4.
# Source: https://llm.mlc.ai/docs/install/mlc_llm.html and PyPI listing.
# If Kaggle moves to CUDA 12.8 swap to mlc-{llm,ai}-nightly-cu128.
!pip install -q --upgrade pip
!pip install -q --pre -U -f https://mlc.ai/wheels \
    mlc-llm-nightly-cu122 \
    mlc-ai-nightly-cu122
# huggingface_hub: leave version unpinned to avoid resolver conflicts.
!pip install -q huggingface_hub


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")


In [ ]:
# Pull the merged HF weights. Two paths:
#   (A) you uploaded /kaggle/working/merged to HF as virgilvox/conduyt-pilot-1.5b-merged
#       (recommended); set MERGED_REPO below and snapshot_download.
#   (B) you're in the same kernel session as train.ipynb and /kaggle/working/merged exists.
from huggingface_hub import snapshot_download

MERGED_REPO = "virgilvox/conduyt-pilot-1.5b-merged"
MERGED_DIR  = "/kaggle/working/merged"

if not os.path.exists(MERGED_DIR):
    snapshot_download(
        repo_id   = MERGED_REPO,
        local_dir = MERGED_DIR,
        token     = os.environ["HF_TOKEN"],
    )
print("merged weights at", MERGED_DIR)


In [ ]:
# Convert weights to q4f16_1 quantization. q4f16_1 is the standard WebLLM
# quantization for browser inference (4-bit weights, fp16 activations).
MLC_OUT = "/kaggle/working/mlc-q4f16_1"
!mkdir -p {MLC_OUT}

!mlc_llm convert_weight \
    {MERGED_DIR} \
    --quantization q4f16_1 \
    --output {MLC_OUT}


In [ ]:
# Generate the model config. --conv-template qwen2 covers Qwen2.5-Coder-1.5B-Instruct.
!mlc_llm gen_config \
    {MERGED_DIR} \
    --quantization q4f16_1 \
    --conv-template qwen2 \
    --output {MLC_OUT}


In [ ]:
# Push to HF Hub.
from huggingface_hub import HfApi, create_repo

MLC_REPO = "virgilvox/conduyt-pilot-1.5b-MLC"
api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(MLC_REPO, exist_ok=True, token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path = MLC_OUT,
    repo_id     = MLC_REPO,
    repo_type   = "model",
)
print("pushed to https://huggingface.co/" + MLC_REPO)


## WebLLM `appConfig` snippet

Drop the snippet below into your client-side WebLLM bootstrap. The `model` URL points at the HF repo this notebook just uploaded; `model_lib` points at the prebuilt WASM library that matches Qwen2.5-Coder-1.5B-Instruct + q4f16_1.

The `model_lib` URL was confirmed against `mlc-ai/web-llm` v0.2.82's `src/config.ts`; it lives in `binary-mlc-llm-libs` under `web-llm-models/v0_2_80/`.

```ts
import { CreateMLCEngine } from '@mlc-ai/web-llm'

const appConfig = {
  model_list: [
    {
      model_id:  'conduyt-pilot-1.5b',
      model:     'https://huggingface.co/virgilvox/conduyt-pilot-1.5b-MLC',
      model_lib: 'https://raw.githubusercontent.com/mlc-ai/binary-mlc-llm-libs/main/web-llm-models/v0_2_80/Qwen2-1.5B-Instruct-q4f16_1-ctx4k_cs1k-webgpu.wasm',
    },
  ],
}

const engine = await CreateMLCEngine('conduyt-pilot-1.5b', { appConfig })
const reply  = await engine.chat.completions.create({
  messages: [{ role: 'user', content: 'Blink an LED on a Pico W.' }],
})
console.log(reply.choices[0].message.content)
```

Replace `virgilvox/...` with your HF user when you publish.
